# Encrypted Quipu Test 54 — Named multi-drop

Releases keys for notebooks 51 (AES raw key) and 52 (ECIES session) in **one** keydrop quipu with TWO named entries. Paid from apocrypha. Run notebooks 51 and 52 first; this reads their saved manifests.

## How this protocol works

A **keydrop** is a plaintext inscription that releases AES keys for previously-sealed encrypted quipus. The body is unencrypted; the whole point is to make the keys publicly readable.

**Canonical body shape (one variant, always):**

```
c1dd 0001  0e  <tone>  0d  00       header (8 bytes)
<count:2 uint16 BE>                   number of drops
for each drop:
    <namelen:1>                       name length (0..255)
    <name:namelen>                    UTF-8 (empty = anonymous, not citeable)
    <ref_txid:32>                     target encrypted quipu txid (raw bytes)
    <key:32>                          the 32-byte AES (or ECIES session) key
[|TITLE|]                              optional outer batch label
```

No distinction between "single" and "multi" keydrop. A single-drop keydrop is just `count = 1`. Every drop has a `<namelen>` byte; an empty name (namelen=0) makes that drop anonymous (released but not citable by name).

**This notebook** inscribes ONE keydrop quipu releasing TWO named keys in a single diamond:

- **`<<keydrop_txid>><<AES message>>`** → releases the AES key from notebook 51
- **`<<keydrop_txid>><<ECIES letter>>`** → releases the ECIES session key from notebook 52

After the inscription confirms, the final cell demonstrates the `<<txid>><<name>>` citation pattern: `resolve_ref()` walks the keydrop, finds the named entry, and returns `(ref_txid, key)` ready for downstream decryption.

**Atomic disclosure.** Both keys appear on chain in one block (once the join confirms). Compare to two separate single-drop inscriptions where the order of broadcast/confirmation isn't guaranteed. The named multi-drop is the right tool for "I'm releasing these together as a coherent act."

**Tone.** The single tone byte (offset 5) applies to the batch as a whole. We use `TONE_REVERENCE` here as a marker for the act of disclosure. For mixed-tone releases (some reverence, some ordinary) you'd inscribe separate keydrops.

**Payer**: apocrypha (single-key).

## Setup

In [1]:
import warnings
warnings.filterwarnings('ignore', message='urllib3 v2 only supports OpenSSL')

import os, sys, json, time
REPO = os.path.abspath('..')
sys.path.insert(0, REPO)
sys.path.insert(0, os.path.join(REPO, 'canonical'))

import cryptos
import colegio_tools as ct
from colegio_tools import _txid_of_serial
from text import build_text_quipu, read_text_quipu
from encrypted import build_keydrop_quipu, read_encrypted_quipu, TONE_ORDINARY, TONE_AFFECTION, TONE_REVERENCE
from quipu_refs import resolve_ref

doge = cryptos.Doge()
TIP_SINGLE = 5_000_000
FEE_PER_KB = 20_000_000

def scaled_fee(draft_hex_str, floor_sat):
    return max(floor_sat, ((len(draft_hex_str) // 2) * FEE_PER_KB) // 1000)

In [2]:
LLAVES = os.path.abspath('../../cinv/llaves')
INSCRIPTIONS_READY = os.path.join(REPO, 'inscriptions_ready')

def load_priv(name, password=''):
    enc = open(os.path.join(LLAVES, f'{name}_prv.enc'), 'rb').read()
    return ct.import_privKey_from_bytes(enc, password)

priv_apo = load_priv('mi')
addr_apo = doge.privtoaddr(priv_apo.to_hex()[2:])
print(f'apocrypha payer: {addr_apo}')

apocrypha payer: D6zKNnkupqRbkB9p5rwix8QiobQWJazjyX


## Load saved keys + target manifests from NB 51 and NB 52

In [4]:
aes_manifest   = json.load(open(os.path.join(INSCRIPTIONS_READY, 'aes_test_manifest.json')))
ecies_manifest = json.load(open(os.path.join(INSCRIPTIONS_READY, 'ecies_single_manifest.json')))
aes_key       = open(aes_manifest['aes_key_path'], 'rb').read()
ecies_session = open(ecies_manifest['session_key_path'], 'rb').read()
print(f'AES target root:   {aes_manifest["root_txid"]}')
print(f'  AES key (hex):   {aes_key.hex()}')
print(f'ECIES target root: {ecies_manifest["root_txid"]}')
print(f'  session (hex):   {ecies_session.hex()}')

AES target root:   f7a8ee4f997f33682038192d1d378eec7260770588411bcd7fe8bebb6be3fa02
  AES key (hex):   427623f3ca03f89c0e8ac9e9b6849a1f1f7eb3029bd0effcc5f191351c9f5c60
ECIES target root: 1bbc2dffbb40e1a93467a123a9af9890d8bb37253e420b7af0392d9f239ba818
  session (hex):   c053ff89ee7a2e0b55d019fd5a35cf37a259698d6e7aa24a3ea6784daaba5866


## Build the named multi-drop

In [5]:
drops = [
    ('AES message',  aes_manifest['root_txid'],   aes_key),
    ('ECIES letter', ecies_manifest['root_txid'], ecies_session),
]
outer_h, outer_b = build_keydrop_quipu(
    drops,
    title='Disclosure for tests 51 and 52',
    tone=TONE_REVERENCE,
)
print(f'header ({len(outer_h)} B): {outer_h.hex()}')
print(f'body   ({len(outer_b)} B):')
import struct
count = struct.unpack(">H", outer_b[:2])[0]
print(f'  count: {count}')
p = 2
for i in range(count):
    nl = outer_b[p]; p += 1
    name = outer_b[p:p+nl].decode(); p += nl
    ref = outer_b[p:p+32].hex(); p += 32
    key = outer_b[p:p+32]; p += 32
    print(f'  drop {i}: namelen={nl}, name={name!r}, ref={ref[:16]}…, key={key.hex()[:16]}…')
if p < len(outer_b):
    print(f'  trailing title: {outer_b[p:].decode("utf-8", errors="replace")}')

header (8 B): c1dd00010eff0d00
body   (187 B):
  count: 2
  drop 0: namelen=11, name='AES message', ref=f7a8ee4f997f3368…, key=427623f3ca03f89c…
  drop 1: namelen=12, name='ECIES letter', ref=1bbc2dffbb40e1a9…, key=c053ff89ee7a2e0b…
  trailing title: |Disclosure for tests 51 and 52|


## Inscribe — single diamond from apocrypha

In [6]:
N_BODY_STRANDS = 4
chunk = len(outer_b) // N_BODY_STRANDS
extra = len(outer_b) %  N_BODY_STRANDS
body_parts, i = [], 0
for k in range(N_BODY_STRANDS):
    sz = chunk + (1 if k < extra else 0)
    body_parts.append(outer_b[i:i+sz]); i += sz
strand_payloads = [outer_h] + body_parts
utxos = ct.rpc_request('listunspent', [0, 9999999, [addr_apo]])
seed_inputs = [{'output': f"{u['txid']}:{u['vout']}", 'value': int(round(u['amount']*1e8))} for u in utxos]
total = sum(s['value'] for s in seed_inputs)
print(f'{len(seed_inputs)} UTXO(s), total {total/1e8:.4f} DOGE')

1 UTXO(s), total 12.0792 DOGE


## Phase I — root tx with scaled fee

In [7]:
priv_hex = priv_apo.to_hex()[2:]
n = len(strand_payloads)
draft_per = (total - TIP_SINGLE) // n
draft = doge.mktx(seed_inputs, [{'value': draft_per, 'address': addr_apo} for _ in range(n)])
doge.signall(draft, priv_hex)
root_fee = scaled_fee(cryptos.serialize(draft), TIP_SINGLE)
print(f'root scaled fee: {root_fee/1e8:.4f} DOGE')
per = (total - root_fee) // n
remainder = (total - root_fee) - per * n
strand_seeds = [per] * n; strand_seeds[0] += remainder
root_tx = doge.mktx(seed_inputs, [{'value': s, 'address': addr_apo} for s in strand_seeds])
doge.signall(root_tx, priv_hex)
root_hex = cryptos.serialize(root_tx)
root_txid = _txid_of_serial(root_hex)
assert ct.rpc_request('sendrawtransaction', [root_hex]) == root_txid
print(f'root_txid: {root_txid}')

root scaled fee: 0.0718 DOGE
root_txid: c277cd570bf36cdca153b95ee90d2d35d2b3d69619394d1f2c3d0d17b855f66c


In [8]:
# Wait for root to confirm
print(f'waiting for root to confirm... ({root_txid[:16]}…)')
start_h = ct.rpc_request('getblockcount')
while True:
    h = ct.rpc_request('getblockcount')
    if h > start_h:
        info = ct.rpc_request('getrawtransaction', [root_txid, 1])
        confs = info.get('confirmations', 0)
        print(f'  {time.strftime("%H:%M:%S")}  block {h}  root confs: {confs}')
        if confs >= 1:
            print(f'✓ root confirmed in block {info.get("blockhash","?")}')
            break
        start_h = h
    time.sleep(15)

waiting for root to confirm... (c277cd570bf36cdc…)
  10:55:25  block 6214480  root confs: 0
  10:57:25  block 6214481  root confs: 0
  10:58:40  block 6214482  root confs: 0
  10:59:26  block 6214483  root confs: 0
  10:59:41  block 6214484  root confs: 1
✓ root confirmed in block 1011747e1822d4c930e7872add7ae7d5c856f9150866b3466239758ee51df96f


## Phase II — strand chains

In [9]:
strands = []
for si, payload in enumerate(strand_payloads):
    cad = ct.CadenaAtom(prvkey=priv_hex, data=payload,
                         utxo_dct={'output': f'{root_txid}:{si}', 'value': strand_seeds[si]},
                         tip=TIP_SINGLE)
    cad.precompute(); strands.append(cad)
    print(f'  strand {si}: {len(cad.txns)} knots')
for si, cad in enumerate(strands):
    for hex_tx, txid in zip(cad.txns, cad.txn_ids):
        assert ct.rpc_request('sendrawtransaction', [hex_tx]) == txid
    print(f'  strand {si} broadcast')

  strand 0: 1 knots
  strand 1: 1 knots
  strand 2: 1 knots
  strand 3: 1 knots
  strand 4: 1 knots
  strand 0 broadcast
  strand 1 broadcast
  strand 2 broadcast
  strand 3 broadcast
  strand 4 broadcast


In [10]:
# Wait for strand termini to confirm
print('waiting for strand termini...')
start_h = ct.rpc_request('getblockcount')
termini = [c.txn_ids[-1] for c in strands]
while True:
    h = ct.rpc_request('getblockcount')
    if h > start_h:
        confs = [ct.rpc_request('getrawtransaction', [t, 1]).get('confirmations', 0) for t in termini]
        print(f'  block {h}  ' + '  '.join(f's{i}:{c}' for i,c in enumerate(confs)))
        if all(c >= 1 for c in confs):
            print('✓ strands confirmed'); break
        start_h = h
    time.sleep(15)

waiting for strand termini...
  block 6214557  s0:0  s1:0  s2:0  s3:0  s4:0
  block 6214558  s0:0  s1:0  s2:0  s3:0  s4:0
  block 6214559  s0:0  s1:0  s2:0  s3:0  s4:0
  block 6214560  s0:0  s1:0  s2:0  s3:0  s4:0
  block 6214561  s0:0  s1:0  s2:0  s3:0  s4:0
  block 6214562  s0:0  s1:0  s2:0  s3:0  s4:0
  block 6214564  s0:0  s1:0  s2:0  s3:0  s4:0
  block 6214565  s0:0  s1:0  s2:0  s3:0  s4:0
  block 6214566  s0:0  s1:0  s2:0  s3:0  s4:0
  block 6214568  s0:0  s1:0  s2:0  s3:0  s4:0
  block 6214569  s0:0  s1:0  s2:0  s3:0  s4:0
  block 6214570  s0:0  s1:0  s2:0  s3:0  s4:0
  block 6214571  s0:0  s1:0  s2:0  s3:0  s4:0
  block 6214572  s0:0  s1:0  s2:0  s3:0  s4:0
  block 6214573  s0:0  s1:0  s2:0  s3:0  s4:0
  block 6214575  s0:0  s1:0  s2:0  s3:0  s4:0
  block 6214576  s0:0  s1:0  s2:0  s3:0  s4:0
  block 6214577  s0:0  s1:0  s2:0  s3:0  s4:0
  block 6214578  s0:0  s1:0  s2:0  s3:0  s4:0
  block 6214579  s0:0  s1:0  s2:0  s3:0  s4:0
  block 6214580  s0:0  s1:0  s2:0  s3:0  s4:0
  bl

## Phase III — join tx with scaled fee

In [11]:
join_inputs = [{'output': f'{c.txn_ids[-1]}:0',
                'value': strand_seeds[si] - TIP_SINGLE * len(c.txns)}
               for si, c in enumerate(strands)]
join_total = sum(i['value'] for i in join_inputs)
draft = doge.mktx(join_inputs, [{'value': join_total - TIP_SINGLE, 'address': addr_apo}])
doge.signall(draft, priv_hex)
join_fee = scaled_fee(cryptos.serialize(draft), TIP_SINGLE)
print(f'join scaled fee: {join_fee/1e8:.4f} DOGE')
join_tx = doge.mktx(join_inputs, [{'value': join_total - join_fee, 'address': addr_apo}])
doge.signall(join_tx, priv_hex)
join_hex = cryptos.serialize(join_tx)
join_txid = _txid_of_serial(join_hex)
assert ct.rpc_request('sendrawtransaction', [join_hex]) == join_txid
print(f'join_txid: {join_txid}')

join scaled fee: 0.1886 DOGE
join_txid: a58e482a51adafc623887d1ff387b4bdac6755f8f2272c161124a661a1961b9f


In [ ]:
# Wait for join to confirm
print(f'waiting for join to confirm... ({join_txid[:16]}…)')
start_h = ct.rpc_request('getblockcount')
while True:
    h = ct.rpc_request('getblockcount')
    if h > start_h:
        info = ct.rpc_request('getrawtransaction', [join_txid, 1])
        confs = info.get('confirmations', 0)
        print(f'  {time.strftime("%H:%M:%S")}  block {h}  join confs: {confs}')
        if confs >= 1:
            print(f'✓ join confirmed'); break
        start_h = h
    time.sleep(15)

## Read back from chain + parse the keydrop

In [12]:
spender_map = {}
for si, cad in enumerate(strands):
    spender_map[f'{root_txid}:{si}'] = cad.txn_ids[0]
    for ki in range(len(cad.txn_ids) - 1):
        spender_map[f'{cad.txn_ids[ki]}:0'] = cad.txn_ids[ki+1]
def walk(start):
    out, cur = '', start
    while True:
        n = spender_map.get(cur)
        if not n: return out
        raw = ct.rpc_request('getrawtransaction', [n, 1])
        op = next((ct.extract_op_return(v) for v in raw['vout'] if ct.extract_op_return(v)), None)
        if not op: return out
        out += op; cur = f'{n}:0'
rec_h = bytes.fromhex(walk(f'{root_txid}:0'))
rec_b = b''.join(bytes.fromhex(walk(f'{root_txid}:{si}')) for si in range(1, len(strands)))
assert rec_h == outer_h and rec_b == outer_b
print('✓ recovered byte-identical')

parsed = read_encrypted_quipu(rec_h, rec_b)
print(f"sub_name: {parsed['sub_name']}, tone: 0x{parsed['tone']:02x}, title: {parsed['title']!r}")
for i, d in enumerate(parsed['drops']):
    print(f'  drop {i}: name={d["name"]!r}, ref_txid={d["ref_txid"][:16]}…, key={d["key"].hex()[:16]}…')
assert parsed['drops'][0]['name'] == 'AES message'
assert parsed['drops'][0]['ref_txid'] == aes_manifest['root_txid']
assert parsed['drops'][0]['key'] == aes_key
assert parsed['drops'][1]['name'] == 'ECIES letter'
assert parsed['drops'][1]['ref_txid'] == ecies_manifest['root_txid']
assert parsed['drops'][1]['key'] == ecies_session
print('✓ both drops match what was inscribed')

✓ recovered byte-identical
sub_name: drop, tone: 0xff, title: 'Disclosure for tests 51 and 52'
  drop 0: name='AES message', ref_txid=f7a8ee4f997f3368…, key=427623f3ca03f89c…
  drop 1: name='ECIES letter', ref_txid=1bbc2dffbb40e1a9…, key=c053ff89ee7a2e0b…
✓ both drops match what was inscribed


## Demonstrate `<<txid>><<name>>` citation resolution

Using `resolve_ref(keydrop_txid, drop_name, fetcher)` — the standard citation primitive. The fetcher returns the inscription's bytes; here we use our local copy (in a real deployed reader, the fetcher would walk the diamond from a node).

In [ ]:
import sys
sys.path.insert(0, '/Users/anthonyschultz/Desktop/Colegio_Invisible')
from colegio_tools import fetch_quipu_bytes

r1 = resolve_ref(root_txid, 'AES message', fetch_quipu_bytes)
print(f"<<{root_txid[:8]}…>><<AES message>>:")
print(f'  kind:      {r1["kind"]}')
print(f'  ref_txid:  {r1["ref_txid"]}')
print(f'  key (hex): {r1["key"].hex()}')
print(f'  parent_title: {r1["parent_title"]!r}')
print()

r2 = resolve_ref(root_txid, 'ECIES letter', fetch_quipu_bytes)
print(f"<<{root_txid[:8]}…>><<ECIES letter>>:")
print(f'  kind:      {r2["kind"]}')
print(f'  ref_txid:  {r2["ref_txid"]}')
print(f'  key (hex): {r2["key"].hex()}')

try:
    resolve_ref(root_txid, 'nonexistent', fetch_quipu_bytes)
except ValueError as e:
    print(f'\n✓ missing name correctly rejected: {str(e)[:100]}')


## End-to-end: use the resolved key to decrypt the AES target

Closes the loop: citation `<<keydrop_txid>><<AES message>>` → resolves to `(target_ref_txid, key)` → fetch target → decrypt with released key. The workflow a future reader follows when walking an essay that cites this keydrop.

In [ ]:
from quipu_refs import resolve_and_decrypt

r = resolve_and_decrypt(root_txid, 'AES message', fetch_quipu_bytes)
inner = read_text_quipu(r['inner_header'], r['inner_body'])
print(f'✓ AES target decrypted via dropped key')
print(f'  recovered title: {inner["title"]!r}')
print(f'  recovered tone:  0x{inner["tone"]:02x}')
print(f'  recovered body:  {inner["body"]!r}')


In [ ]:
# Without the keydrop, only the envelope recipients could read this. With the
# dropped session key, anyone can — resolve_and_decrypt auto-dispatches on the
# target's sub-family (0xae→key, 0xec→session_key).
r = resolve_and_decrypt(root_txid, 'ECIES letter', fetch_quipu_bytes)
inner = read_text_quipu(r['inner_header'], r['inner_body'])
print(f'✓ ECIES target decrypted via dropped session key')
print(f'  recovered title: {inner["title"]!r}')
print(f'  recovered tone:  0x{inner["tone"]:02x}')
print(f'  recovered body:  {inner["body"]!r}')


## Save manifest

In [22]:
json.dump({'keydrop_root_txid': root_txid,
           'keydrop_join_txid': join_txid,
           'drops': [
               {'name': 'AES message',  'releases': aes_manifest['root_txid']},
               {'name': 'ECIES letter', 'releases': ecies_manifest['root_txid']},
           ],
           'payer': addr_apo},
          open(os.path.join(INSCRIPTIONS_READY, 'keydrops_manifest.json'), 'w'), indent=2)
print('keydrop manifest saved')

keydrop manifest saved
